In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model
from scipy.spatial.distance import pdist

actions = [
    'pipe', 'tiger', 'landmarks'
]
seq_length = 30

model = load_model('models/model.h5')

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5)

cap = cv2.VideoCapture(0)

seq = []
action_seq = []

while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    img = cv2.flip(img, 1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = hands.process(img_rgb)
    img = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    if result.multi_hand_landmarks is not None:
        for res in result.multi_hand_landmarks:
            joint = np.zeros((21, 4))
            for j, lm in enumerate(res.landmark):
                joint[j] = [lm.x, lm.y, lm.z, lm.visibility]

            # 관절 각도 계산
            v1 = joint[[0,1,2,3,0,5,6,7,0,9,10,11,0,13,14,15,0,17,18,19], :3]
            v2 = joint[[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20], :3]
            v = v2 - v1
            v = v / np.linalg.norm(v, axis=1)[:, np.newaxis]
            angle = np.arccos(np.einsum('nt,nt->n',
                v[[0,1,2,4,5,6,8,9,10,12,13,14,16,17,18],:],
                v[[1,2,3,5,6,7,9,10,11,13,14,15,17,18,19],:]))
            angle = np.degrees(angle)

            # 랜드마크 간 거리 계산 (추가 feature)
            xyz = joint[:, :3]
            distances = pdist(xyz)
            
            # feature 수가 198이 되도록 조정 (joint.flatten():84, angle:15, distances[:99]:99)
            d = np.concatenate([joint.flatten(), angle, distances[:99]])
            seq.append(d)

            mp_drawing.draw_landmarks(img, res, mp_hands.HAND_CONNECTIONS)

            if len(seq) < seq_length:
                continue

            input_data = np.expand_dims(np.array(seq[-seq_length:], dtype=np.float32), axis=0)

            y_pred = model.predict(input_data).squeeze()
            print(f"Predictions: {y_pred}")  # 디버깅 출력

            i_pred = int(np.argmax(y_pred))
            conf = y_pred[i_pred]
            print(f"Predicted action: {actions[i_pred]}, Confidence: {conf:.2f}")  # 디버깅 출력

            if conf < 0.8:
                continue

            action = actions[i_pred]
            action_seq.append(action)
            print(f"Action sequence: {action_seq}")  # 디버깅 출력

            if len(action_seq) < 3:
                continue

            this_action = '?'
            if action_seq[-1] == action_seq[-2] == action_seq[-3]:
                this_action = action

            cv2.putText(img, f'{this_action.upper()}',
                        org=(int(res.landmark[0].x * img.shape[1]), int(res.landmark[0].y * img.shape[0] + 20)),
                        fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(255, 255, 255), thickness=2)

    cv2.imshow('img', img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np
import pygame
import mediapipe as mp

# MediaPipe Hands 초기화
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# 이미지 로드 및 알파 채널 제거 (RGB로 변환)
saxophone_img = cv2.imread('pipe.jpg', cv2.IMREAD_UNCHANGED)
if saxophone_img.shape[2] == 4:
    saxophone_img = cv2.cvtColor(saxophone_img, cv2.COLOR_RGBA2RGB)
tiger_paw_img = cv2.imread('tiger.png', cv2.IMREAD_UNCHANGED)
if tiger_paw_img.shape[2] == 4:
    tiger_paw_img = cv2.cvtColor(tiger_paw_img, cv2.COLOR_RGBA2RGB)
landmark_imgs = []
for i in range(5):
    img = cv2.imread(f'landmark_{i}.png', cv2.IMREAD_UNCHANGED)
    if img is not None and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    landmark_imgs.append(img)

# 사운드 초기화
pygame.mixer.init()
sax_sound = pygame.mixer.Sound('pipe.mp3')
tiger_sound = pygame.mixer.Sound('tiger.mp3')
landmark_sounds = [pygame.mixer.Sound('landmark.mp3') for i in range(5)]

def apply_effect(img, action, hand_landmarks):
    if action == 'pipe':
        x, y = int(hand_landmarks[0].x * img.shape[1]), int(hand_landmarks[0].y * img.shape[0])
        if y + 100 <= img.shape[0] and x + 100 <= img.shape[1]:
            saxophone_img_resized = cv2.resize(saxophone_img, (100, 100))
            img[y:y+100, x:x+100] = cv2.addWeighted(img[y:y+100, x:x+100], 0.5, saxophone_img_resized, 0.5, 0)
            sax_sound.play()
    elif action == 'tiger':
        x, y = int(hand_landmarks[0].x * img.shape[1]), int(hand_landmarks[0].y * img.shape[0])
        if y + 100 <= img.shape[0] and x + 100 <= img.shape[1]:
            tiger_paw_img_resized = cv2.resize(tiger_paw_img, (100, 100))
            img[y:y+100, x:x+100] = cv2.addWeighted(img[y:y+100, x:x+100], 0.5, tiger_paw_img_resized, 0.5, 0)
            tiger_sound.play()
    elif action == 'landmarks':
        for i in range(5):
            x, y = int(hand_landmarks[8+i].x * img.shape[1]), int(hand_landmarks[8+i].y * img.shape[0])
            if i < len(landmark_imgs) and landmark_imgs[i] is not None and y + 50 <= img.shape[0] and x + 50 <= img.shape[1]:
                landmark_img_resized = cv2.resize(landmark_imgs[i], (50, 50))
                img[y:y+50, x:x+50] = cv2.addWeighted(img[y:y+50, x:x+50], 0.5, landmark_img_resized, 0.5, 0)
                landmark_sounds[i].play()

# 웹캠 초기화
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("웹캠을 열 수 없습니다.")
    exit()

action_seq = []
actions = ['pipe', 'tiger', 'landmarks']

while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    img = cv2.flip(img, 1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    img = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            this_action = actions[0]  # 실제로는 분류 결과에 따라 바꿔야 함
            apply_effect(img, this_action, hand_landmarks.landmark)

    cv2.imshow('img', img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
from tensorflow.keras.models import load_model

model = load_model('models/model1.h5')  # 모델 파일 경로에 맞게 수정


In [ ]:
import numpy as np
from scipy.spatial.distance import pdist

def extract_features(hand_landmarks):
    joint = np.zeros((21, 4))
    for j, lm in enumerate(hand_landmarks.landmark):
        joint[j] = [lm.x, lm.y, lm.z, lm.visibility]

    # 관절 각도 계산
    v1 = joint[[0,1,2,3,0,5,6,7,0,9,10,11,0,13,14,15,0,17,18,19], :3]
    v2 = joint[[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20], :3]
    v = v2 - v1
    v = v / np.linalg.norm(v, axis=1)[:, np.newaxis]
    angle = np.arccos(np.einsum('nt,nt->n',
        v[[0,1,2,4,5,6,8,9,10,12,13,14,16,17,18],:],
        v[[1,2,3,5,6,7,9,10,11,13,14,15,17,18,19],:]))
    angle = np.degrees(angle)

    # 랜드마크 간 거리 계산 (추가 feature)
    xyz = joint[:, :3]
    distances = pdist(xyz)
    # 모델 입력 차원에 맞게 조정 (예: 198)
    d = np.concatenate([joint.flatten(), angle, distances[:99]])
    return d


In [ ]:
import cv2
import numpy as np
import pygame
import mediapipe as mp

# MediaPipe Hands 초기화
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)

# 이미지 로드 및 알파 채널 제거 (RGB로 변환)
saxophone_img = cv2.imread('pipe.jpg', cv2.IMREAD_UNCHANGED)
if saxophone_img.shape[2] == 4:
    saxophone_img = cv2.cvtColor(saxophone_img, cv2.COLOR_RGBA2RGB)
tiger_paw_img = cv2.imread('tiger.png', cv2.IMREAD_UNCHANGED)
if tiger_paw_img.shape[2] == 4:
    tiger_paw_img = cv2.cvtColor(tiger_paw_img, cv2.COLOR_RGBA2RGB)
landmark_imgs = []
for i in range(5):
    img = cv2.imread(f'landmark_{i}.png', cv2.IMREAD_UNCHANGED)
    if img is not None and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    landmark_imgs.append(img)

# 사운드 초기화
pygame.mixer.init()
sax_sound = pygame.mixer.Sound('pipe.mp3')
tiger_sound = pygame.mixer.Sound('tiger.mp3')
landmark_sounds = [pygame.mixer.Sound('landmark.mp3') for i in range(5)]

def apply_effect(img, action, hand_landmarks):
    if action == 'pipe':
        x, y = int(hand_landmarks[0].x * img.shape[1]), int(hand_landmarks[0].y * img.shape[0])
        if y + 100 <= img.shape[0] and x + 100 <= img.shape[1]:
            saxophone_img_resized = cv2.resize(saxophone_img, (100, 100))
            img[y:y+100, x:x+100] = cv2.addWeighted(img[y:y+100, x:x+100], 0.5, saxophone_img_resized, 0.5, 0)
            sax_sound.play()
    elif action == 'tiger':
        x, y = int(hand_landmarks[0].x * img.shape[1]), int(hand_landmarks[0].y * img.shape[0])
        if y + 100 <= img.shape[0] and x + 100 <= img.shape[1]:
            tiger_paw_img_resized = cv2.resize(tiger_paw_img, (100, 100))
            img[y:y+100, x:x+100] = cv2.addWeighted(img[y:y+100, x:x+100], 0.5, tiger_paw_img_resized, 0.5, 0)
            tiger_sound.play()
    elif action == 'landmarks':
        for i in range(5):
            x, y = int(hand_landmarks[8+i].x * img.shape[1]), int(hand_landmarks[8+i].y * img.shape[0])
            if i < len(landmark_imgs) and landmark_imgs[i] is not None and y + 50 <= img.shape[0] and x + 50 <= img.shape[1]:
                landmark_img_resized = cv2.resize(landmark_imgs[i], (50, 50))
                img[y:y+50, x:x+50] = cv2.addWeighted(img[y:y+50, x:x+50], 0.5, landmark_img_resized, 0.5, 0)
                landmark_sounds[i].play()

# 웹캠 초기화
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("웹캠을 열 수 없습니다.")
    exit()

actions = ['pipe', 'tiger', 'landmarks']
seq = []
action_seq = []
seq_length = 30  # 모델 입력 시퀀스 길이

while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    img = cv2.flip(img, 1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    img = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            features = extract_features(hand_landmarks)
            seq.append(features)

            if len(seq) < seq_length:
                continue

            input_data = np.expand_dims(np.array(seq[-seq_length:], dtype=np.float32), axis=0)
            y_pred = model.predict(input_data).squeeze()
            i_pred = int(np.argmax(y_pred))
            conf = y_pred[i_pred]
            action = actions[i_pred]
            action_seq.append(action)

            if len(action_seq) < 3:
                continue

            this_action = '?'
            if action_seq[-1] == action_seq[-2] == action_seq[-3]:
                this_action = action

            # 화면에 동작 이름 출력
            cv2.putText(img, f'{this_action.upper()} ({conf:.2f})',
                        org=(int(hand_landmarks.landmark[0].x * img.shape[1]),
                             int(hand_landmarks.landmark[0].y * img.shape[0] + 20)),
                        fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(255, 255, 255), thickness=2)

            # 이펙트 적용
            apply_effect(img, this_action, hand_landmarks.landmark)

    cv2.imshow('img', img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np
import pygame #소리부분
import mediapipe as mp
from scipy.spatial.distance import pdist

# MediaPipe Hands 초기화
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# 이미지 로드 및 알파 채널 제거 (RGB로 변환)
saxophone_img = cv2.imread('pipe.jpg', cv2.IMREAD_UNCHANGED)
if saxophone_img.shape[2] == 4:
    saxophone_img = cv2.cvtColor(saxophone_img, cv2.COLOR_RGBA2RGB)
tiger_paw_img = cv2.imread('tiger.png', cv2.IMREAD_UNCHANGED)
if tiger_paw_img.shape[2] == 4:
    tiger_paw_img = cv2.cvtColor(tiger_paw_img, cv2.COLOR_RGBA2RGB)
landmark_imgs = []
for i in range(5):
    img = cv2.imread(f'landmark_{i}.png', cv2.IMREAD_UNCHANGED)
    if img is not None and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    landmark_imgs.append(img)

# 사운드 초기화 #소리부분
pygame.mixer.init()
sax_sound = pygame.mixer.Sound('pipe.mp3')
tiger_sound = pygame.mixer.Sound('tiger.mp3')
landmark_sounds = [pygame.mixer.Sound('landmark.mp3') for i in range(5)]


def extract_features(hand_landmarks):
    joint = np.zeros((21, 4))
    for j, lm in enumerate(hand_landmarks.landmark):
        joint[j] = [lm.x, lm.y, lm.z, lm.visibility]

    # 관절 각도 계산
    v1 = joint[[0,1,2,3,0,5,6,7,0,9,10,11,0,13,14,15,0,17,18,19], :3]
    v2 = joint[[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20], :3]
    v = v2 - v1
    v = v / np.linalg.norm(v, axis=1)[:, np.newaxis]
    angle = np.arccos(np.einsum('nt,nt->n',
        v[[0,1,2,4,5,6,8,9,10,12,13,14,16,17,18],:],
        v[[1,2,3,5,6,7,9,10,11,13,14,15,17,18,19],:]))
    angle = np.degrees(angle)

    # 랜드마크 간 거리 계산 (추가 feature)
    xyz = joint[:, :3]
    distances = pdist(xyz)
    # 모델 입력 차원에 맞게 조정 (예: 198)
    d = np.concatenate([joint.flatten(), angle, distances[:99]])
    return d

def apply_effect(img, action, hand_landmarks):
    if action == 'pipe':
        x, y = int(hand_landmarks[0].x * img.shape[1]), int(hand_landmarks[0].y * img.shape[0])
        if y + 100 <= img.shape[0] and x + 100 <= img.shape[1]:
            saxophone_img_resized = cv2.resize(saxophone_img, (100, 100))
            img[y:y+100, x:x+100] = cv2.addWeighted(img[y:y+100, x:x+100], 0.5, saxophone_img_resized, 0.5, 0)
            sax_sound.play()#소리 추가부분
    elif action == 'tiger':
        x, y = int(hand_landmarks[0].x * img.shape[1]), int(hand_landmarks[0].y * img.shape[0])
        if y + 100 <= img.shape[0] and x + 100 <= img.shape[1]:
            tiger_paw_img_resized = cv2.resize(tiger_paw_img, (100, 100))
            img[y:y+100, x:x+100] = cv2.addWeighted(img[y:y+100, x:x+100], 0.5, tiger_paw_img_resized, 0.5, 0)
            tiger_sound.play()
    elif action == 'landmarks':
        for i in range(5):
            x, y = int(hand_landmarks[8+i].x * img.shape[1]), int(hand_landmarks[8+i].y * img.shape[0])
            if i < len(landmark_imgs) and landmark_imgs[i] is not None and y + 50 <= img.shape[0] and x + 50 <= img.shape[1]:
                landmark_img_resized = cv2.resize(landmark_imgs[i], (50, 50))
                img[y:y+50, x:x+50] = cv2.addWeighted(img[y:y+50, x:x+50], 0.5, landmark_img_resized, 0.5, 0)
                landmark_sounds[i].play()

# 학습된 모델 불러오기
from tensorflow.keras.models import load_model
model = load_model('models/model1.h5')

# 웹캠 초기화
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("웹캠을 열 수 없습니다.")
    exit()

actions = ['pipe', 'tiger', 'landmarks']  # 모델 출력 클래스 수와 반드시 일치해야 함
seq = []
action_seq = []
seq_length = 30  # 모델 입력 시퀀스 길이

while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    img = cv2.flip(img, 1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    img = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            features = extract_features(hand_landmarks)
            seq.append(features)

            if len(seq) < seq_length:
                continue

            input_data = np.expand_dims(np.array(seq[-seq_length:], dtype=np.float32), axis=0)
            y_pred = model.predict(input_data).squeeze()
            i_pred = int(np.argmax(y_pred))

            # actions 리스트 범위 체크 (안전하게 처리)
            if 0 <= i_pred < len(actions):
                action = actions[i_pred]
                action_seq.append(action)
                print(f"Predicted: {action}, Confidence: {y_pred[i_pred]:.2f}")
            else:
                print(f"Warning: Invalid prediction index {i_pred} (max: {len(actions)-1})")
                action = "unknown"

            if len(action_seq) < 3:
                continue

            this_action = '?'
            if action_seq[-1] == action_seq[-2] == action_seq[-3]:
                this_action = action

            # 화면에 동작 이름 출력
            cv2.putText(img, f'{this_action.upper()} ({y_pred[i_pred]:.2f})',
                        org=(int(hand_landmarks.landmark[0].x * img.shape[1]),
                             int(hand_landmarks.landmark[0].y * img.shape[0] + 20)),
                        fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(255, 255, 255), thickness=2)

            # 이펙트 적용 (actions 범위 내 인덱스만)
            if action in actions:
                apply_effect(img, this_action, hand_landmarks.landmark)

    cv2.imshow('img', img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


이거 좋음

In [ ]:
import cv2
import numpy as np
import pygame
import mediapipe as mp

# MediaPipe Hands 초기화
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# 호랑이 발바닥 이미지 로드 (알파 채널 포함)
tiger_paw_img = cv2.imread('tigerpaw.png', cv2.IMREAD_UNCHANGED)

# 사운드 초기화
pygame.mixer.init()
tiger_sound = pygame.mixer.Sound('tiger.mp3')

def get_palm_center(hand_landmarks, img_shape):
    palm_indices = [0, 5, 9, 13, 17]
    x = int(np.mean([hand_landmarks[i].x for i in palm_indices]) * img_shape[1])
    y = int(np.mean([hand_landmarks[i].y for i in palm_indices]) * img_shape[0])
    return x, y

def get_hand_openness(hand_landmarks, img_shape):
    # 중지 끝(8)과 손바닥 중심 거리로 오므림 정도 측정
    x_c, y_c = get_palm_center(hand_landmarks, img_shape)
    x_f = int(hand_landmarks[8].x * img_shape[1])
    y_f = int(hand_landmarks[8].y * img_shape[0])
    distance = np.sqrt((x_c - x_f) ** 2 + (y_c - y_f) ** 2)
    return distance

def overlay_png(bg_img, fg_img, x, y):
    h, w = fg_img.shape[:2]
    if x < 0 or y < 0 or x + w > bg_img.shape[1] or y + h > bg_img.shape[0]:
        return bg_img
    if fg_img.shape[2] == 4:
        alpha_fg = fg_img[:, :, 3] / 255.0
        alpha_bg = 1.0 - alpha_fg
        for c in range(3):
            bg_img[y:y+h, x:x+w, c] = (alpha_fg * fg_img[:, :, c] +
                                       alpha_bg * bg_img[y:y+h, x:x+w, c])
    else:
        bg_img[y:y+h, x:x+w] = fg_img
    return bg_img

def apply_tiger_effect(img, hand_landmarks):
    x, y = get_palm_center(hand_landmarks, img.shape)
    openness = get_hand_openness(hand_landmarks, img.shape)
    # 최소 60, 최대 180 픽셀로 크기 조절
    min_size, max_size = 60, 180
    # 거리값을 60~180 사이로 정규화 (적절히 조정 가능)
    size = int(np.clip(openness * 1.8, min_size, max_size))
    tiger_resized = cv2.resize(tiger_paw_img, (size, size))
    x1, y1 = int(x - size // 2), int(y - size // 2)
    overlay_png(img, tiger_resized, x1, y1)
    tiger_sound.play()

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("웹캠을 열 수 없습니다.")
    exit()

while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    img = cv2.flip(img, 1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    img = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            apply_tiger_effect(img, hand_landmarks.landmark)

    cv2.imshow('Tiger Hand Effect', img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


최종 호랑이

In [ ]:
import cv2
import numpy as np
import pygame
import mediapipe as mp
import os

# 이미지와 사운드 파일 체크
if not os.path.exists('tigerpaw.png'):
    print("tigerpaw.png 파일이 없습니다.")
    exit()
if not os.path.exists('tiger.mp3'):
    print("tiger.mp3 파일이 없습니다.")
    exit()

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

tigerpaw_img = cv2.imread('tigerpaw.png', cv2.IMREAD_UNCHANGED)
if tigerpaw_img is None:
    print("tigerpaw.png 이미지를 불러올 수 없습니다.")
    exit()

pygame.mixer.init()
tiger_sound = pygame.mixer.Sound('tiger.mp3')

def get_palm_center(hand_landmarks, img_shape):
    palm_indices = [0, 5, 9, 13, 17]
    x = int(np.mean([hand_landmarks[i].x for i in palm_indices]) * img_shape[1])
    y = int(np.mean([hand_landmarks[i].y for i in palm_indices]) * img_shape[0])
    return x, y

def get_hand_openness(hand_landmarks, img_shape):
    x_c, y_c = get_palm_center(hand_landmarks, img_shape)
    x_f = int(hand_landmarks[8].x * img_shape[1])
    y_f = int(hand_landmarks[8].y * img_shape[0])
    distance = np.sqrt((x_c - x_f) ** 2 + (y_c - y_f) ** 2)
    return distance

def overlay_png(bg_img, fg_img, x, y):
    h, w = fg_img.shape[:2]
    if x < 0 or y < 0 or x + w > bg_img.shape[1] or y + h > bg_img.shape[0]:
        return
    if fg_img.shape[2] == 4:
        alpha_fg = fg_img[:, :, 3] / 255.0
        alpha_bg = 1.0 - alpha_fg
        for c in range(3):
            bg_img[y:y+h, x:x+w, c] = (alpha_fg * fg_img[:, :, c] +
                                       alpha_bg * bg_img[y:y+h, x:x+w, c])
    else:
        bg_img[y:y+h, x:x+w] = fg_img

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("웹캠을 열 수 없습니다.")
    exit()

sound_played = False

while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    img = cv2.flip(img, 1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    img = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            x, y = get_palm_center(hand_landmarks.landmark, img.shape)
            openness = get_hand_openness(hand_landmarks.landmark, img.shape)
            min_size, max_size = 80, 180
            size = int(np.clip(openness * 1.8, min_size, max_size))
            tigerpaw_resized = cv2.resize(tigerpaw_img, (size, size))
            x1, y1 = int(x - size // 2), int(y - size // 2)
            overlay_png(img, tigerpaw_resized, x1, y1)
            if not sound_played:
                tiger_sound.play()
                sound_played = True
    else:
        sound_played = False

    cv2.imshow('Tiger Hand Effect', img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


피리 효과

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import pygame
import os

# 파일 체크
if not os.path.exists('pipe.png'):
    print("pipe.png 파일이 없습니다.")
    exit()
if not os.path.exists('pipe.mp3'):
    print("pipe.mp3 파일이 없습니다.")
    exit()

# MediaPipe 초기화
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# 파이프 이미지 로드
pipe_img = cv2.imread('pipe.png', cv2.IMREAD_UNCHANGED)
if pipe_img is None:
    print("pipe.png 이미지를 불러올 수 없습니다.")
    exit()

# 사운드 초기화
pygame.mixer.init()
pipe_sound = pygame.mixer.Sound('pipe.mp3')

def get_hand_center(hand_landmarks, img_shape):
    # 손바닥 중심 좌표 계산
    palm_indices = [0, 5, 9, 13, 17]
    x = np.mean([hand_landmarks[i].x for i in palm_indices]) * img_shape[1]
    y = np.mean([hand_landmarks[i].y for i in palm_indices]) * img_shape[0]
    return np.array([x, y])

def get_extreme_points(hand_landmarks_list, img_shape):
    all_thumbs = []
    all_pinkies = []
    
    for hand_landmarks in hand_landmarks_list:
        thumb = np.array([hand_landmarks[4].x * img_shape[1], hand_landmarks[4].y * img_shape[0]])
        pinky = np.array([hand_landmarks[20].x * img_shape[1], hand_landmarks[20].y * img_shape[0]])
        
        all_thumbs.append(thumb)
        all_pinkies.append(pinky)
    
    top_thumb = min(all_thumbs, key=lambda x: x[1])
    bottom_pinky = max(all_pinkies, key=lambda x: x[1])
    
    return top_thumb, bottom_pinky

def overlay_pipe_simple(img, top_point, bottom_point, pipe_img, is_flipped=False):
    h, w = img.shape[:2]
    
    # 파이프 이미지 좌우 반전
    current_pipe = cv2.flip(pipe_img, 1) if is_flipped else pipe_img
    
    # 두 점 사이 거리 계산
    distance = np.linalg.norm(bottom_point - top_point)
    
    # 파이프 이미지 크기 조정 (세로로만)
    scale = distance / current_pipe.shape[0]
    new_width = int(current_pipe.shape[1] * scale)
    new_height = int(distance)
    
    if new_width < 1 or new_height < 1:
        return
    
    resized_pipe = cv2.resize(current_pipe, (new_width, new_height))
    
    # 두 점의 중점에 파이프 배치 (회전 없이)
    mid_x = int((top_point[0] + bottom_point[0]) / 2)
    mid_y = int((top_point[1] + bottom_point[1]) / 2)
    
    start_x = mid_x - new_width // 2
    start_y = mid_y - new_height // 2
    
    # 오버레이 영역 계산
    y1 = max(0, start_y)
    y2 = min(h, start_y + new_height)
    x1 = max(0, start_x)
    x2 = min(w, start_x + new_width)
    
    # 파이프 이미지에서의 해당 영역
    pipe_y1 = max(0, -start_y)
    pipe_y2 = pipe_y1 + (y2 - y1)
    pipe_x1 = max(0, -start_x)
    pipe_x2 = pipe_x1 + (x2 - x1)
    
    # 알파 채널 처리
    if resized_pipe.shape[2] == 4 and (y2 > y1) and (x2 > x1):
        pipe_region = resized_pipe[pipe_y1:pipe_y2, pipe_x1:pipe_x2]
        alpha = pipe_region[:, :, 3] / 255.0 * 0.5
        alpha_3d = np.stack([alpha] * 3, axis=2)
        
        img[y1:y2, x1:x2] = (alpha_3d * pipe_region[:, :, :3] + 
                            (1 - alpha_3d) * img[y1:y2, x1:x2]).astype(np.uint8)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("웹캠을 열 수 없습니다.")
    exit()

# 손 움직임 감지 변수
prev_hand_center = None
movement_threshold = 30  # 픽셀 단위 움직임 임계값
is_pipe_flipped = False
sound_played = False

while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    img = cv2.flip(img, 1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    img = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    if results.multi_hand_landmarks and len(results.multi_hand_landmarks) >= 2:
        # 손 랜드마크 그리기
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        
        hand_landmarks_list = [hand_landmarks.landmark for hand_landmarks in results.multi_hand_landmarks]
        
        # 현재 손의 중심 위치 계산 (첫 번째 손 기준)
        current_hand_center = get_hand_center(hand_landmarks_list[0], img.shape)
        
        # 손 움직임 감지
        if prev_hand_center is not None:
            movement_distance = np.linalg.norm(current_hand_center - prev_hand_center)
            if movement_distance > movement_threshold:
                is_pipe_flipped = not is_pipe_flipped  # 반전 상태 토글
                print(f"손 움직임 감지! 거리: {movement_distance:.2f}, 반전: {is_pipe_flipped}")
        
        prev_hand_center = current_hand_center
        
        # 파이프 효과 적용
        top_thumb, bottom_pinky = get_extreme_points(hand_landmarks_list, img.shape)
        overlay_pipe_simple(img, top_thumb, bottom_pinky, pipe_img, is_pipe_flipped)
        
        # 사운드 재생
        if not sound_played:
            pipe_sound.play()
            sound_played = True
    else:
        sound_played = False
        prev_hand_center = None

    # 현재 상태 표시
    status_text = f"Pipe Flipped: {is_pipe_flipped}"
    cv2.putText(img, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    cv2.imshow('Hand Pipe Flip Effect', img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import pygame
import os

# 사운드 파일 하나만 사용
sound_file = 'landmark.mp3'
if not os.path.exists(sound_file):
    print(f"{sound_file} 파일이 없습니다.")
    exit()

# MediaPipe 초기화
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

# Pygame 사운드 초기화
pygame.mixer.init()
sound = pygame.mixer.Sound(sound_file)

# 이미지 5개 로드
landmark_imgs = []
for i in range(5):
    img_path = f'landmark_{i}.png'
    if not os.path.exists(img_path):
        print(f"{img_path} 파일이 없습니다.")
        landmark_imgs.append(None)
    else:
        img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        landmark_imgs.append(img)

current_index = 0
cooldown = 0  # 연속 발동 방지

def overlay_effect(img, overlay_img, x, y):
    if overlay_img is None:
        return img
    h, w = overlay_img.shape[:2]
    y1, y2 = max(0, y), min(img.shape[0], y + h)
    x1, x2 = max(0, x), min(img.shape[1], x + w)
    oy1, oy2 = max(0, -y), h - max(0, y + h - img.shape[0])
    ox1, ox2 = max(0, -x), w - max(0, x + w - img.shape[1])
    if y2 - y1 <= 0 or x2 - x1 <= 0:
        return img
    if overlay_img.shape[2] == 4:
        alpha = overlay_img[oy1:oy2, ox1:ox2, 3] / 255.0
        alpha = np.repeat(alpha[:, :, np.newaxis], 3, axis=2)
        overlay_rgb = overlay_img[oy1:oy2, ox1:ox2, :3]
        img[y1:y2, x1:x2] = (1 - alpha) * img[y1:y2, x1:x2] + alpha * overlay_rgb
    else:
        img[y1:y2, x1:x2] = overlay_img[oy1:oy2, ox1:ox2]
    return img

cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_frame)
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # 검지 끝(8번) 좌표
            index_tip = hand_landmarks.landmark[8]
            cx, cy = int(index_tip.x * frame.shape[1]), int(index_tip.y * frame.shape[0])
            # 엄지 끝(4번) 좌표
            thumb_tip = hand_landmarks.landmark[4]
            distance = np.hypot(index_tip.x - thumb_tip.x, index_tip.y - thumb_tip.y)
            # 포인팅 제스처 감지 (엄지-검지 가까움)
            if distance < 0.05 and cooldown == 0:
                img = landmark_imgs[current_index]
                if img is not None:
                    h, w = img.shape[:2]
                    frame = overlay_effect(frame, img, cx - w//2, cy - h//2)
                sound.play()
                current_index = (current_index + 1) % 5
                cooldown = 20  # 쿨다운
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
    if cooldown > 0:
        cooldown -= 1
    cv2.imshow('Landmark Pointing Effect', frame)
    if cv2.waitKey(1) == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import pygame
import os
import time

# 사운드 파일 하나만 사용
sound_file = 'landmark.mp3'
if not os.path.exists(sound_file):
    print(f"{sound_file} 파일이 없습니다.")
    exit()

# 이미지 파일 체크 및 로드 (크기 고정)
image_size = 150
landmark_imgs = []
for i in range(5):
    img_path = f'landmark_{i}.png'
    if not os.path.exists(img_path):
        print(f"{img_path} 파일이 없습니다.")
        landmark_imgs.append(None)
    else:
        img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        if img is not None:
            img = cv2.resize(img, (image_size, image_size))
            if img.shape[2] == 3:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
        landmark_imgs.append(img)

# MediaPipe 초기화
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

# Pygame 사운드 초기화
pygame.mixer.init()
sound = pygame.mixer.Sound(sound_file)

# 활성화된 이펙트 관리 (인덱스, 위치, 만료시간)
active_effects = []

current_index = 0
last_activation = 0
cooldown = 1.0  # 1초 쿨다운
effect_duration = 2.0  # 2초간 유지

def overlay_image(img, overlay_img, x, y):
    if overlay_img is None:
        return
    h, w = overlay_img.shape[:2]
    # 이미지 하단 중앙을 (x, y)에 맞춤
    y = int(y - h)
    x = int(x - w // 2)
    # 경계 검사
    y1 = max(0, y)
    y2 = min(img.shape[0], y + h)
    x1 = max(0, x)
    x2 = min(img.shape[1], x + w)
    oy1, oy2 = max(0, -y), h - max(0, y + h - img.shape[0])
    ox1, ox2 = max(0, -x), w - max(0, x + w - img.shape[1])
    if y2 <= y1 or x2 <= x1:
        return
    overlay_region = overlay_img[oy1:oy2, ox1:ox2]
    if overlay_region.shape[2] == 4:
        alpha = overlay_region[:, :, 3] / 255.0
        alpha = np.repeat(alpha[:, :, np.newaxis], 3, axis=2)
        overlay_rgb = overlay_region[:, :, :3]
        img[y1:y2, x1:x2] = (1 - alpha) * img[y1:y2, x1:x2] + alpha * overlay_rgb

def landmark_pointing_effect():
    global current_index, last_activation
    cap = cv2.VideoCapture(0)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)
        current_time = time.time()

        # 활성화된 이펙트 렌더링
        for idx, (img_idx, pos, end_time) in enumerate(list(active_effects)):
            if current_time < end_time:
                overlay_image(frame, landmark_imgs[img_idx], pos[0], pos[1])
            else:
                active_effects.pop(idx)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # 검지 끝 좌표
                index_tip = hand_landmarks.landmark[8]
                cx = int(index_tip.x * frame.shape[1])
                cy = int(index_tip.y * frame.shape[0])
                # 엄지 끝 좌표
                thumb_tip = hand_landmarks.landmark[4]
                distance = np.hypot(index_tip.x - thumb_tip.x, index_tip.y - thumb_tip.y)
                if distance < 0.05 and (current_time - last_activation) > cooldown:
                    sound.play()
                    active_effects.append((current_index, (cx, cy), current_time + effect_duration))
                    current_index = (current_index + 1) % 5
                    last_activation = current_time
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        cv2.imshow('Landmark Effect', frame)
        if cv2.waitKey(1) == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

if __name__ == '__main__':
    landmark_pointing_effect()
